In [ ]:
# ---- 0. 环境设置 -----------------------------------------------------------
Sys.setenv(SP_EVOLUTION_STATUS = "2")

set.seed(1)
library(SeuratObject)
library(Matrix)
library(dplyr)
library(tidyr)
library(readr)
library(Seurat)

In [ ]:
# ---- 0. 参数设置 ------------------------------------------------------------
date <- "2026-07-06_1"
data_dir <- "/project/work/2026-07-06_development_DEG/Data/"
min_cells <- 30
max_cell <- Inf

In [ ]:
# ---- 1. 读取数据 ------------------------------------------------------------
cat("=== 1. 读取 Seurat 对象 ===\n")
obj <- readRDS("/project/work/2026-05-25_create_summary/Data/2026-06-05_develop_all_human_harmony_anno.rds")
cat("  细胞数:", ncol(obj), "\n")
cat("  基因数:", nrow(obj), "\n")
cat("  细胞类型:", paste(levels(CONTACT_EMAIL$celltype), collapse = ", "), "\n")

In [ ]:
meta <- readRDS("/project/work/2026-05-25_create_summary/Data/2026-06-05_develop_all_human_harmony_anno_meta.rds")
CONTACT_EMAIL <- meta

In [ ]:
# ---- 2. 添加 stage 列 -------------------------------------------------------
cat("\n=== 2. 定义发育阶段 ===\n")
# CONTACT_EMAIL <- CONTACT_EMAIL %>%
#   mutate(
#     stage = case_when(
#       donor_age %in% c("70days", "79days", "87days", "91days") ~ "Early",
#       donor_age %in% c("100days", "103days", "116days") ~ "Mid",
#       donor_age %in% c("137days", "141days", "142days", "162days", "165days") ~ "Late"
#     ),
#     stage = factor(stage, levels = c("Early", "Mid", "Late"))
#   )

In [ ]:
stage_summary <- CONTACT_EMAIL %>%
  count(stage, donor_age, donor_id) %>%
  arrange(stage, donor_age)
cat("  阶段划分:\n")
print(stage_summary)

In [ ]:
# ---- 3. 检查细胞数 ----------------------------------------------------------
cat("\n=== 3. 检查各组合细胞数 ===\n")
cell_counts <- CONTACT_EMAIL %>%
  count(celltype, donor_age, stage, tissue, donor_id, name = "n_cells") %>%
  arrange(celltype, stage, donor_age, tissue)
head(cell_counts)

In [ ]:
# 保存细胞数报告
write.csv(cell_counts, paste0(data_dir, date, "_cell_counts_by_group.csv"), row.names = FALSE)
cat("  细胞数报告已保存: ", paste0(data_dir, date, "_cell_counts_by_group.csv"), "\n")

In [ ]:
# 标记不足组合（< min_cells 细胞）
low_count <- cell_counts %>% filter(n_cells < min_cells)
if (nrow(low_count) > 0) {
  cat("  警告: 以下组合细胞数 < ", min_cells, ", 差异分析可能不可靠:\n")
  print(low_count)
} else {
  cat("  所有组合细胞数均 >=  ", min_cells, "\n")
}

In [ ]:
# ---- 4. 构建 Regional Pseudobulk (celltype × donor × stage × tissue) --------
cat("\n=== 4. 构建 Regional Pseudobulk ===\n")
celltypes <- levels(CONTACT_EMAIL$celltype)

In [ ]:
for (ct in celltypes) {
    cat("  处理 celltype:", ct, "\n")

    sub <- subset(obj, subset = celltype == ct)

    if (ncol(sub) == 0) {
      stop("该 celltype 无细胞: ", ct)
      next
    }

    sub$regional_group <- paste("pb", sub$donor_age, sub$stage, sub$tissue, sep = "-")

    # ---- 新增：过滤低细胞数组合 + 全局上限降采样 ----
    meta_df <- CONTACT_EMAIL
    meta_df$cell_id <- rownames(meta_df)
    meta_df <- as_tibble(meta_df)

    # 1. 过滤 min_cells
    meta_df <- meta_df %>%
      group_by(regional_group) %>%
      filter(n() >= min_cells) %>%
      ungroup()

    # 2. 全局上限降采样
    if (nrow(meta_df) > 0) {
      set.seed(1)
      meta_df <- meta_df %>%
        group_by(regional_group) %>%
        sample_n(size = min(n(), max_cell)) %>%
        ungroup()
      sub <- sub[, meta_df$cell_id]
    }
    # ---- 新增结束 ----

    cell_n <- CONTACT_EMAIL %>%
      count(regional_group, name = "n_cells")
  
    agg <- AggregateExpression(
      sub,
      group.by = "regional_group",
      assays = "RNA",
      return.seurat = FALSE,
      verbose = FALSE
    )
    pb_counts <- agg$RNA
    
    colData <- CONTACT_EMAIL %>%
      distinct(regional_group, donor_age, stage, tissue) %>%
      left_join(cell_n, by = "regional_group") %>%
      mutate(
        donor_age = factor(donor_age),
        stage = factor(stage, levels = c("Early", "Mid", "Late")),
        tissue = factor(tissue, levels = c("peripheral", "macula"))
      ) %>%
      select(group = regional_group, donor_age, stage, tissue, n_cells)
    rownames(colData) <- colData$group
    
    colData <- colData[colnames(pb_counts), , drop = FALSE]
    print(all(colnames(pb_counts) == rownames(colData)))
    
    saveRDS(
      list(counts = pb_counts, colData = colData),
      file = paste0(data_dir, date, "_", ct, "_regional_pseudobulk.rds")
    )
    print(paste0(data_dir, date, "_", ct, "_regional_pseudobulk.rds"))

    write.csv(
      colData,
      paste0(data_dir, date, "_", ct, "_regional_colData.csv"),
      row.names = FALSE
    )
    print(paste0(data_dir, date, "_", ct, "_regional_colData.csv"))

    cat("    完成: ", ncol(pb_counts), " 个 pseudobulk 样本\n")
}

In [ ]:
cat("\n========================================\n")
cat("Pseudobulk 构建完成！\n")
cat("========================================\n")